# Hyperparameter Tuning (Optuna)

<div style="text-align: justify">

The following notebook is dedicated to <b>hyperparameter optimisation</b> for the <b>Tau Supersymmetry</b> search analysis. Using <b>Optuna</b> with TPE sampling and median pruning, a k-fold cross-validated search is performed over the hyperparameter space for both XGBoost (BDT) and PyTorch (DNN) classifiers. The best parameters are exported as Hydra-compatible YAML configs for use in subsequent training runs.

</div>

## Pipeline Summary

| Step | Module | Description |
|------|--------|-------------|
| Config | `hydra.compose` | Load analysis, model, and tuning configuration |
| Load | `io.load_dataframe` | Read mc.parquet from feature engineering output |
| Labels | — | Derive ordered class names from `eventOrigin` |
| Prepare | `splits.prepare_features_target` | Separate training features, labels, and weights |
| Split | `splits.kfold_split` | Stratified K-fold split for CV |
| Study | `tuning.create_study` | Create/resume Optuna study with SQLite persistence |
| Optimize | `tuning.bdt_objective` / `tuning.dnn_objective` | K-fold CV objectives with pruning |
| Results | `optuna.visualization` | Optimization history and parameter importance plots |
| Export | `tuning.export_best_params` | Save best params as Hydra-compatible YAML |

The same pipeline is available as a CLI via `python tune.py` or `make tune`.

## Initialization

### Libraries

Configuration:
* [Hydra](https://hydra.cc/)
* [OmegaConf](https://omegaconf.readthedocs.io/)
* [pyrootutils](https://github.com/ashleve/pyrootutils)

Data Processing:
* [Pandas](https://pandas.pydata.org/)
* [NumPy](https://numpy.org/)

Hyperparameter Optimisation:
* [Optuna](https://optuna.readthedocs.io/)

Machine Learning:
* [XGBoost](https://xgboost.readthedocs.io/en/stable/)
* [PyTorch](https://pytorch.org/)
* [scikit-learn](https://scikit-learn.org/stable/)

Data Visualization:
* [Plotly](https://plotly.com/python/) (Optuna visualizations)

### Notebook

Activating autoreload of imported modules.

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

Initializing the project root.

In [ ]:
import pyrootutils

path = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

Suppressing unessential warnings and applying ATLAS style.

In [ ]:
from src.utils import suppress_warnings
from src.visualization.plots import apply_atlas_style

suppress_warnings()
apply_atlas_style()

## Configuration

Loading the Hydra configuration. All analysis parameters (run, region, channel, model hyperparameters, tuning settings) are defined in `configs/` and can be overridden here.

> **Model selection:** change `overrides` to `["model=dnn"]` for DNN tuning.

In [ ]:
from hydra import compose, initialize_config_dir

initialize_config_dir(config_dir=str(path / "configs"), version_base="1.3")
cfg = compose(config_name="config")

# For DNN tuning, uncomment the following:
# cfg = compose(config_name="config", overrides=["model=dnn"])

Resolving input and output directories from config.

In [ ]:
from src.processing.analysis import get_output_paths

output_paths = get_output_paths(cfg)
dataframes_dir = path / output_paths["dataframes_dir"]
models_dir = path / output_paths["models_dir"]

models_dir.mkdir(parents=True, exist_ok=True)

## Data Loading & Preparation

Loading the MC DataFrame produced by the feature engineering pipeline.

In [ ]:
from src.processing.io import load_dataframe

df_mc = load_dataframe(dataframes_dir / "mc.parquet")
print(f"Loaded: {len(df_mc):,} events, {len(df_mc.columns)} columns")

Deriving ordered class names from `eventOrigin`.

In [ ]:
from src.eda.utils import get_class_names

class_names = get_class_names(df_mc)
n_classes = len(class_names)
print(f"Classes ({n_classes}): {class_names}")

Separating training features, integer class labels, and per-event sample weights.

In [ ]:
from src.models.splits import prepare_features_target

X, y, weights = prepare_features_target(df_mc)
print(f"Feature matrix: {X.shape}")

### K-Fold Split

Tuning always uses stratified K-fold cross-validation for a robust generalization estimate.

In [ ]:
from src.models.splits import kfold_split

n_splits = cfg.tuning.n_splits
folds = kfold_split(X, y, weights, n_splits=n_splits, seed=cfg.seed)
print(f"K-fold: {n_splits} stratified folds")

## Optuna Study

Creating or resuming the Optuna study. Studies are persisted in SQLite for automatic resumability.

In [ ]:
from src.models.tuning import create_study

storage_path = models_dir / cfg.tuning.storage_filename
study = create_study(cfg, storage_path)
print(f"Study: {study.study_name}")
print(f"Existing trials: {len(study.trials)}")

## Optimization

### BDT (XGBoost)

Running the BDT objective with XGBoost pruning callbacks. Skip this cell if tuning the DNN.

In [ ]:
from src.models.tuning import bdt_objective

if cfg.model.name == "xgboost":
    study.optimize(
        lambda trial: bdt_objective(trial, cfg, folds, n_classes),
        n_trials=cfg.tuning.n_trials,
    )
    print(f"Optimization complete: {len(study.trials)} trials")
else:
    print("Skipping BDT tuning (model is not xgboost)")

### DNN (PyTorch)

Running the DNN objective with epoch-level pruning. Skip this cell if tuning the BDT.

In [ ]:
from src.models.dnn import resolve_device
from src.models.tuning import dnn_objective

if cfg.model.name == "pytorch_dnn":
    device = resolve_device()
    print(f"Device: {device}")

    study.optimize(
        lambda trial: dnn_objective(trial, cfg, folds, n_classes, device),
        n_trials=cfg.tuning.n_trials,
    )
    print(f"Optimization complete: {len(study.trials)} trials")
else:
    print("Skipping DNN tuning (model is not pytorch_dnn)")

## Results

### Best Trial Summary

In [ ]:
import optuna

n_completed = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
n_pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])

print(f"Completed: {n_completed} | Pruned: {n_pruned}")

if n_completed > 0:
    print(f"\nBest trial #{study.best_trial.number}")
    print(f"  Value: {study.best_trial.value:.6f}")
    print(f"  Params:")
    for k, v in study.best_trial.params.items():
        print(f"    {k}: {v}")

### Optimization History

In [ ]:
from optuna.visualization import plot_optimization_history

fig = plot_optimization_history(study)
fig.show()

### Parameter Importances

In [ ]:
from optuna.visualization import plot_param_importances

if n_completed >= 2:
    fig = plot_param_importances(study)
    fig.show()
else:
    print("Need at least 2 completed trials for parameter importance analysis")

## Export Best Parameters

Exporting the best trial's parameters as a Hydra-compatible YAML model config. This file can be copied to `configs/model/` and used for training:

```bash
cp <output_path> configs/model/xgboost_tuned.yaml
python train.py model=xgboost_tuned
```

In [ ]:
from src.models.tuning import export_best_params

if n_completed > 0:
    suffix = "xgboost" if cfg.model.name == "xgboost" else "dnn"
    params_path = models_dir / f"{suffix}_best_params.yaml"
    export_best_params(study, cfg.model.name, cfg.model, params_path)
    print(f"Best params exported to: {params_path}")
else:
    print("No completed trials — nothing to export")